<a href="https://colab.research.google.com/github/aroffender/Foodie-Network/blob/main/cse428_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets, preprocessing, linear_model
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical, plot_model
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive

In [4]:
# Mount the Google Drive at /content/drive
drive.mount('/content/drive')

# Verify by listing the files in the drive
!ls /content/drive/My\ Drive/

Mounted at /content/drive
 02_20201027_Lab03.pdf
 04_21301122_Experiment_No_02.pdf
'06_20201167_Assignment2 (other).pdf'
 06_20201167_Assignment4.pdf
 11_21301258_Experiment_No_02.pdf
 11_21301258_graded_quiz-1.pdf
 11_21301258_lab_03.pdf
'11_21301258_Md. Ismam Hossain Redwan.pdf'
 166C8609-E959-4723-8E62-87BCBB3FA3C9.jpeg
 1.docx
 1.gdoc
 1-l7uMN3JaKN5hXaRnJLkDt9K9G2RlBVh
'21301258_Md. Ismam Hossain Redwan_21301258_psy101.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment01_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment02_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan.gdoc'
'321 ass 2.gdoc'
'A1_21301258_Md. Ismam Hossain Redwan_sec-11.pdf'
 Add.gdoc
'after peraphrase.gdoc'
'Amar fasi chai.pdf'
'CamScanner 03-02-2022 22.18 (1).pdf'
'Colab Notebooks'
'Copy of Company links.gsheet'
'Copy of Experiment No 03-Submission Form.pdf'
'Copy of F&M BOR.gsheet'
'Copy of Quillbot Premium .gdoc'
'Copy of Sponsor List  .gsheet'
' CS

In [5]:
# Verify by listing the files in the drive
!ls /content/drive/My\ Drive/

 02_20201027_Lab03.pdf
 04_21301122_Experiment_No_02.pdf
'06_20201167_Assignment2 (other).pdf'
 06_20201167_Assignment4.pdf
 11_21301258_Experiment_No_02.pdf
 11_21301258_graded_quiz-1.pdf
 11_21301258_lab_03.pdf
'11_21301258_Md. Ismam Hossain Redwan.pdf'
 166C8609-E959-4723-8E62-87BCBB3FA3C9.jpeg
 1.docx
 1.gdoc
 1-l7uMN3JaKN5hXaRnJLkDt9K9G2RlBVh
'21301258_Md. Ismam Hossain Redwan_21301258_psy101.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment01_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment02_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan.gdoc'
'321 ass 2.gdoc'
'A1_21301258_Md. Ismam Hossain Redwan_sec-11.pdf'
 Add.gdoc
'after peraphrase.gdoc'
'Amar fasi chai.pdf'
'CamScanner 03-02-2022 22.18 (1).pdf'
'Colab Notebooks'
'Copy of Company links.gsheet'
'Copy of Experiment No 03-Submission Form.pdf'
'Copy of F&M BOR.gsheet'
'Copy of Quillbot Premium .gdoc'
'Copy of Sponsor List  .gsheet'
' CSE370_Section04_Group09_Fal

In [6]:
import zipfile

# Specify the path to the downloaded ZIP file
zip_file_path = "/content/drive/MyDrive/cse428 project/plantsegv2.zip"

# Specify the directory where you want to extract the dataset
extract_to_directory = "/content/drive/MyDrive/cse428 project"

# Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_directory)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D,concatenate, Dropout, Conv2DTranspose)
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import MeanIoU
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load dataset paths (modify these paths for your dataset)
data_path = "path_to_images/"
mask_path = "path_to_masks/"

# Function to load images and masks
def load_images_and_masks(image_path, mask_path):
    images = []
    masks = []
    for img_file in os.listdir(image_path):
        img = tf.keras.utils.load_img(os.path.join(image_path, img_file), target_size=(128, 128))
        img = tf.keras.utils.img_to_array(img) / 255.0
        images.append(img)

        mask_file = img_file.replace(".jpg", ".png")  # Adjust mask extension if needed
        mask = tf.keras.utils.load_img(os.path.join(mask_path, mask_file), target_size=(128, 128), color_mode="grayscale")
        mask = tf.keras.utils.img_to_array(mask) / 255.0
        masks.append(mask)

    return np.array(images), np.array(masks)

# Load data
images, masks = load_images_and_masks(data_path, mask_path)

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(images, masks, test_size=0.2, random_state=42)

# Define convolutional block
def convolution_func(num_filters, input_layer):
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(input_layer)
    x = Dropout(0.1)(x)
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(x)
    return x

# Define encoder block
def encoder(num_filters, input_layer):
    s = convolution_func(num_filters, input_layer)
    p = MaxPooling2D((2, 2))(s)
    return s, p

# Define decoder block
def decoder(skip_connection, previous_input, num_filters):
    concatenated_input = concatenate([Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(previous_input), skip_connection])
    output = convolution_func(num_filters, concatenated_input)
    return output

# U-Net Model Implementation
def unet_model(input_size=(128, 128, 3)):
    inputs = Input(input_size)

    # Encoder
    s1, p1 = encoder(64, inputs)
    s2, p2 = encoder(128, p1)
    s3, p3 = encoder(256, p2)
    s4, p4 = encoder(512, p3)

    # Bottleneck
    b1 = convolution_func(1024, p4)

    # Decoder
    d1 = decoder(s4, b1, 512)
    d2 = decoder(s3, d1, 256)
    d3 = decoder(s2, d2, 128)
    d4 = decoder(s1, d3, 64)

    # Output
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(d4)

    model = Model(inputs, outputs)
    return model

# Compile the model
model = unet_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=16, epochs=25)

# Evaluate the model
predictions = model.predict(X_val)

# Metrics: IOU, Dice Coefficient, and Pixel Accuracy
def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

def pixel_accuracy(y_true, y_pred):
    y_pred = (y_pred > 0.5).astype(np.float32)
    return np.mean(y_true == y_pred)

mean_iou = MeanIoU(num_classes=2)
mean_iou.update_state(y_val, predictions)
iou_score = mean_iou.result().numpy()
dice_score = dice_coefficient(y_val, predictions)
pixel_acc = pixel_accuracy(y_val, predictions)

print("Mean IOU:", iou_score)
print("Dice Coefficient:", dice_score)
print("Pixel Accuracy:", pixel_acc)

# Visualize results for validation set
for i in range(3):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(X_val[i])
    plt.subplot(1, 3, 2)
    plt.title("Ground Truth")
    plt.imshow(y_val[i].squeeze(), cmap='gray')
    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow(predictions[i].squeeze(), cmap='gray')
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'path_to_images/'

In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, Conv2DTranspose)
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import MeanIoU
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ### Step 1: Set Dataset Paths ###
# Update `base_data_path` to point to your dataset directory that contains the "image" and "annotation" folders.
base_data_path = "/content/drive/MyDrive/cse428 project"  # Replace with the path to your dataset
image_path = os.path.join(base_data_path, "/content/drive/MyDrive/cse428 project/plantsegv2/images")  # Path to the image folder
annotation_path = os.path.join(base_data_path, "/content/drive/MyDrive/cse428 project/plantsegv2/annotations")  # Path to the annotation folder

# ### Step 2: Define Function to Load Data ###
# This function pairs images with corresponding masks, resizes them to (128, 128), and normalizes the pixel values.
def load_data(image_dir, annotation_dir):
    images, masks = [], []  # Initialize lists to store images and masks

    # Iterate over all files in the image directory
    for img_file in os.listdir(image_dir):
        # Load and normalize image
        img = tf.keras.utils.load_img(os.path.join(image_dir, img_file), target_size=(128, 128))
        img = tf.keras.utils.img_to_array(img) / 255.0
        images.append(img)  # Add the image to the list

        # Load and normalize corresponding mask
        mask_file = img_file  # Ensure filenames match for image and mask
        mask = tf.keras.utils.load_img(os.path.join(annotation_dir, mask_file),target_size=(128, 128), color_mode="grayscale")
        mask = tf.keras.utils.img_to_array(mask) / 255.0
        masks.append(mask)  # Add the mask to the list

    return np.array(images), np.array(masks)

# ### Step 3: Load Training, Validation, and Test Data ###
# Call `load_data` for each dataset split (train, val, test). Ensure your dataset contains these subfolders.
X_train, y_train = load_data(os.path.join(image_path, "train"), os.path.join(annotation_path, "train"))
X_val, y_val = load_data(os.path.join(image_path, "val"), os.path.join(annotation_path, "val"))
X_test, y_test = load_data(os.path.join(image_path, "test"), os.path.join(annotation_path, "test"))

# Print the shape of the loaded data to confirm correct loading.
print(f"Train data: {X_train.shape}, {y_train.shape}")
print(f"Validation data: {X_val.shape}, {y_val.shape}")
print(f"Test data: {X_test.shape}, {y_test.shape}")

# ### Step 4: Define U-Net Model ###
# Implement U-Net from scratch with encoder, bottleneck, and decoder blocks.
def convolution_block(inputs, num_filters):
    """Convolution block with two Conv2D layers, ReLU activation, and dropout."""
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(inputs)
    x = Dropout(0.1)(x)
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(x)
    return x

def encoder_block(inputs, num_filters):
    """Encoder block: two convolutions + max pooling."""
    x = convolution_block(inputs, num_filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters):
    """Decoder block: transposed convolution + concatenation + convolutions."""
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    x = concatenate([x, skip_features])  # Add skip connection
    x = convolution_block(x, num_filters)
    return x

def build_unet(input_shape):
    """Assemble U-Net model with encoder, bottleneck, and decoder blocks."""
    inputs = Input(input_shape)

    # Encoder
    s1, p1 = encoder_block(inputs, 64)  # First encoder block
    s2, p2 = encoder_block(p1, 128)    # Second encoder block
    s3, p3 = encoder_block(p2, 256)    # Third encoder block
    s4, p4 = encoder_block(p3, 512)    # Fourth encoder block

    # Bottleneck
    b1 = convolution_block(p4, 1024)   # Bottleneck layer

    # Decoder
    d1 = decoder_block(b1, s4, 512)    # First decoder block
    d2 = decoder_block(d1, s3, 256)    # Second decoder block
    d3 = decoder_block(d2, s2, 128)    # Third decoder block
    d4 = decoder_block(d3, s1, 64)     # Fourth decoder block

    # Output layer: 1x1 convolution with sigmoid activation
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(d4)

    return Model(inputs, outputs)

# ### Step 5: Compile and Train the Model ###
# Build the U-Net model and compile with binary crossentropy loss and accuracy metric.
model = build_unet((128, 128, 3))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model with the training and validation datasets.
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=16, epochs=25)

# ### Step 6: Evaluate the Model ###
# Predict masks for the test dataset.
predictions = model.predict(X_test)

# Define metrics
def dice_coefficient(y_true, y_pred):
    """Calculate Dice Coefficient."""
    y_pred = (y_pred > 0.5).astype(np.float32)  # Threshold predictions
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

def pixel_accuracy(y_true, y_pred):
    """Calculate pixel accuracy."""
    y_pred = (y_pred > 0.5).astype(np.float32)  # Threshold predictions
    return np.mean(y_true == y_pred)

# Calculate and display metrics
mean_iou = MeanIoU(num_classes=2)
mean_iou.update_state(y_test, (predictions > 0.5).astype(np.float32))
iou_score = mean_iou.result().numpy()
dice_score = dice_coefficient(y_test, predictions)
pixel_acc = pixel_accuracy(y_test, predictions)

print("Mean IOU:", iou_score)
print("Dice Coefficient:", dice_score)
print("Pixel Accuracy:", pixel_acc)

# ### Step 7: Visualize Results ###
# Visualize a few predictions along with their ground truth and input images.
for i in range(3):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(X_test[i])
    plt.subplot(1, 3, 2)
    plt.title("Ground Truth")
    plt.imshow(y_test[i].squeeze(), cmap='gray')
    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow(predictions[i].squeeze(), cmap='gray')
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/cse428 project/plantsegv2/annotations/train/zucchini_powdery_mildew_Bing_0090.jpg'

In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, Conv2DTranspose)
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import MeanIoU
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import zipfile



# Step 2: Define paths for images and masks
# Update the paths based on the dataset structure
image_base_path = "/content/drive/MyDrive/cse428 project/images"  # Adjust this path
mask_base_path = "/content/drive/MyDrive/cse428 project/masks"    # Adjust this path

# Define subdirectories for train, val, and test splits
subdirs = ["train", "val", "test"]

# Step 3: Function to load images and masks
# This function reads images and corresponding masks, ensuring alignment by filenames
def load_images_and_masks(image_base_path, mask_base_path, subdirs):
    images = []
    masks = []

    for subdir in subdirs:  # Loop through train, val, and test directories
        image_path = os.path.join(image_base_path, subdir)
        mask_path = os.path.join(mask_base_path, subdir)

        if not os.path.exists(image_path):  # Check if image directory exists
            print(f"Image directory does not exist: {image_path}")
            continue
        if not os.path.exists(mask_path):  # Check if mask directory exists
            print(f"Mask directory does not exist: {mask_path}")
            continue

        for img_file in os.listdir(image_path):  # Loop through each image file
            try:
                # Load the image and normalize pixel values to [0, 1]
                img = tf.keras.utils.load_img(os.path.join(image_path, img_file), target_size=(128, 128))
                img = tf.keras.utils.img_to_array(img) / 255.0
                images.append(img)

                # Replace image extension with mask extension to find corresponding mask
                mask_file = img_file.replace(".jpg", ".png")  # Adjust extensions as needed
                mask_full_path = os.path.join(mask_path, mask_file)

                if not os.path.exists(mask_full_path):  # Check if corresponding mask exists
                    print(f"Mask file missing: {mask_full_path}")
                    continue

                # Load the mask and normalize pixel values to [0, 1]
                mask = tf.keras.utils.load_img(mask_full_path, target_size=(128, 128), color_mode="grayscale")
                mask = tf.keras.utils.img_to_array(mask) / 255.0
                masks.append(mask)

            except Exception as e:
                # Handle any errors that occur during file processing
                print(f"Error processing file {img_file}: {e}")

    return np.array(images), np.array(masks)

# Step 4: Load Training, Validation, and Test Data
# Call `load_images_and_masks` for each dataset split
X_train, y_train = load_images_and_masks(image_base_path, mask_base_path, ["train"])
X_val, y_val = load_images_and_masks(image_base_path, mask_base_path, ["val"])
X_test, y_test = load_images_and_masks(image_base_path, mask_base_path, ["test"])

# Step 5: Define the U-Net model
# This is the U-Net architecture for segmentation
def convolution_func(num_filters, input_layer):
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(input_layer)
    x = Dropout(0.1)(x)
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(x)
    return x

def encoder(num_filters, input_layer):
    s = convolution_func(num_filters, input_layer)
    p = MaxPooling2D((2, 2))(s)
    return s, p

def decoder(skip_connection, previous_input, num_filters):
    concatenated_input = concatenate([Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(previous_input), skip_connection])
    output = convolution_func(num_filters, concatenated_input)
    return output

def unet_model(input_size=(128, 128, 3)):
    inputs = Input(input_size)

    # Encoder
    s1, p1 = encoder(64, inputs)
    s2, p2 = encoder(128, p1)
    s3, p3 = encoder(256, p2)
    s4, p4 = encoder(512, p3)

    # Bottleneck
    b1 = convolution_func(1024, p4)

    # Decoder
    d1 = decoder(s4, b1, 512)
    d2 = decoder(s3, d1, 256)
    d3 = decoder(s2, d2, 128)
    d4 = decoder(s1, d3, 64)

    # Output
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(d4)

    model = Model(inputs, outputs)
    return model

# Step 6: Compile and Train the Model
model = unet_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=16, epochs=25)

# Step 7: Evaluate the Model
# Predict on validation set and compute metrics
predictions = model.predict(X_val)

def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

def pixel_accuracy(y_true, y_pred):
    y_pred = (y_pred > 0.5).astype(np.float32)
    return np.mean(y_true == y_pred)

mean_iou = MeanIoU(num_classes=2)
mean_iou.update_state(y_val, predictions)
iou_score = mean_iou.result().numpy()
dice_score = dice_coefficient(y_val, predictions)
pixel_acc = pixel_accuracy(y_val, predictions)

print("Mean IOU:", iou_score)
print("Dice Coefficient:", dice_score)
print("Pixel Accuracy:", pixel_acc)

# Step 8: Visualize Results
for i in range(3):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(X_val[i])
    plt.subplot(1, 3, 2)
    plt.title("Ground Truth")
    plt.imshow(y_val[i].squeeze(), cmap='gray')
    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow(predictions[i].squeeze(), cmap='gray')
    plt.show()


Image directory does not exist: /content/drive/MyDrive/cse428 project/images/train
Image directory does not exist: /content/drive/MyDrive/cse428 project/images/val
Image directory does not exist: /content/drive/MyDrive/cse428 project/images/test
Epoch 1/25


ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data:0", shape=(16,), dtype=float32). Expected shape (None, 128, 128, 3), but input has incompatible shape (16,)[0m

Arguments received by Functional.call():
  • inputs=tf.Tensor(shape=(16,), dtype=float32)
  • training=True
  • mask=None